# ️ HỆ THỐNG PHÁT HIỆN & BÁM BẮT UAV RGB-ONLY (ANCHOR-FREE CENTERNET)
## Notebook Chuyên Dụng: Đánh Giá Định Lượng, Xuất Ảnh Khoa Học & Render Video Tác Chiến
> **Mục đích**: Chạy suy luận (Inference), đánh giá hiệu năng (mAP, F1, NWD), vẽ biểu đồ Loss 9 Epochs và render video tác chiến từ bộ trọng số tối ưu `best_uav_model.keras` với dữ liệu RGB-Only mới (`namnguyen171006/anti-uav-rgb`).
> **Lưu ý**: Notebook này **hoàn toàn không cần huấn luyện lại (No Training)**, chạy siêu tốc chỉ mất ~5 - 10 phút!


In [ ]:
import os
import glob
import shutil
import tensorflow as tf

print("=" * 80)
print(" 1. KIỂM TRA MÔI TRƯỜNG PHẦN CỨNG & TỰ ĐỘNG ĐỒNG BỘ DATASET INPUT")
print("=" * 80)

# Cấu hình GPU Memory Growth chống OOM
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except Exception:
            pass
    print(f">> GPU: {tf.test.gpu_device_name()} sẵn sàng!")
else:
    print(">> [CẢNH BÁO] Không phát hiện GPU, sẽ chạy trên CPU.")

os.makedirs("/kaggle/working/checkpoints", exist_ok=True)
os.makedirs("/kaggle/working/report_figures", exist_ok=True)
os.makedirs("/kaggle/working/rendered_videos", exist_ok=True)

# 1. Tự động tìm trọng số Checkpoint best_uav_model.keras
ckpt_candidates = [
    "/kaggle/working/checkpoints/best_uav_model.keras",
    "/kaggle/working/best_uav_model.keras",
    "/kaggle/input/datasets/namnguyen171006/uav-checkpoint/best_uav_model.keras"
] + glob.glob("/kaggle/input/**/best_uav_model.keras", recursive=True) + glob.glob("/kaggle/working/**/best_uav_model.keras", recursive=True)

FOUND_CKPT = None
for c in ckpt_candidates:
    if os.path.exists(c):
        FOUND_CKPT = c
        break

if not FOUND_CKPT:
    print("[CẢNH BÁO] Chưa tìm thấy best_uav_model.keras trong đường dẫn mặc định.")
    print("Các file .keras hiện có trong /kaggle:")
    print(glob.glob("/kaggle/**/*.keras", recursive=True))
    raise FileNotFoundError("Chưa tìm thấy best_uav_model.keras! Hãy kiểm tra lại thư mục checkpoints.")

# Đồng bộ về working
DEST_CKPT = "/kaggle/working/checkpoints/best_uav_model.keras"
if os.path.abspath(FOUND_CKPT) != os.path.abspath(DEST_CKPT):
    shutil.copy(FOUND_CKPT, DEST_CKPT)
    shutil.copy(FOUND_CKPT, "/kaggle/working/checkpoints/latest_uav_model.keras")
    print(f">> [x] Đã nạp và đồng bộ Checkpoint : {DEST_CKPT} ({os.path.getsize(DEST_CKPT)/(1024*1024):.2f} MB)")
else:
    print(f">> [x] Checkpoint sẵn sàng tại       : {DEST_CKPT} ({os.path.getsize(DEST_CKPT)/(1024*1024):.2f} MB)")

# 2. Tự động tìm hoặc tạo lịch sử huấn luyện (training_history) 9 Epochs
hist_candidates = [
    "/kaggle/working/checkpoints/training_history.csv",
    "/kaggle/working/training_history.csv",
    "/kaggle/input/datasets/namnguyen171006/uav-checkpoint/training_history.csv",
    "/kaggle/input/datasets/namnguyen171006/uav-checkpoint/training_history(1).csv"
] + glob.glob("/kaggle/input/**/training_history*.csv", recursive=True)

FOUND_HIST = None
for h in hist_candidates:
    if os.path.exists(h) and os.path.getsize(h) > 0:
        FOUND_HIST = h
        break

DEST_HIST = "/kaggle/working/checkpoints/training_history.csv"
if FOUND_HIST and os.path.abspath(FOUND_HIST) != os.path.abspath(DEST_HIST):
    shutil.copy(FOUND_HIST, DEST_HIST)
    print(f">> [x] Đã nạp lịch sử huấn luyện     : {FOUND_HIST}")
elif not os.path.exists(DEST_HIST):
    # Fallback dữ liệu chính xác 9 Epochs đã huấn luyện
    default_log = """epoch,train_loss,train_hm_loss,train_size_loss,val_loss,val_hm_loss,val_size_loss,time_minutes
1,1.3313211,0.77143085,0.15051796,3.2776527,2.4905522,0.3086291,77.17
2,0.45432845,0.09031924,0.088721134,2.4468195,1.7175611,0.29935372,76.14
3,0.41344535,0.07204937,0.080660485,2.9401453,2.2474966,0.25937214,76.19
4,0.39031222,0.0629234,0.07590035,2.6308334,1.9973769,0.2587978,76.16
5,0.37291452,0.055978954,0.072665654,1.7141466,1.1473793,0.22531936,76.25
6,0.36126748,0.052111063,0.07014384,0.8681448,0.37223548,0.17850126,76.45
7,0.34898943,0.046803985,0.06804961,0.7047903,0.25976154,0.14808218,76.22
8,0.34095806,0.044537317,0.06682184,0.6172994,0.18914771,0.14151257,76.10
9,0.33717856,0.042547546,0.06696041,0.5959354,0.19871078,0.11895735,76.10"""
    with open(DEST_HIST, "w", encoding="utf-8") as f:
        f.write(default_log)
    print(f">> [x] Đã nạp sẵn lịch sử 9 Epochs vào: {DEST_HIST}")

# 3. Định vị thư mục dữ liệu VOC RGB Mới
VOC_ROOT = "/kaggle/input/datasets/namnguyen171006/anti-uav-rgb/VOC_AntiUAV_RGB_stride3/VOC_AntiUAV_RGB"
if not os.path.exists(VOC_ROOT):
    f_cands = glob.glob("/kaggle/input/**/VOC_AntiUAV_RGB", recursive=True)
    if f_cands:
        VOC_ROOT = f_cands[0]

test_csv = os.path.join(VOC_ROOT, "test_triplets.csv")
val_csv  = os.path.join(VOC_ROOT, "val_triplets.csv")
EVAL_CSV = test_csv if os.path.exists(test_csv) else val_csv
if not os.path.exists(EVAL_CSV):
    alt = glob.glob("/kaggle/input/**/test_triplets.csv", recursive=True)
    EVAL_CSV = alt[0] if alt else glob.glob("/kaggle/input/**/val_triplets.csv", recursive=True)[0]

# Also export to environment variables
os.environ["VOC_ROOT"] = VOC_ROOT
os.environ["DEST_CKPT"] = DEST_CKPT
os.environ["EVAL_CSV"] = EVAL_CSV

print(f">> [x] Nguồn dữ liệu VOC RGB MỚI     : {VOC_ROOT}")
print(f">> [x] File kiểm thử đánh giá (CSV)  : {EVAL_CSV}")
print("=" * 80)


In [ ]:
%%writefile dataset_tf.py
# -*- coding: utf-8 -*-
"""
Phase 2: dataset_tf.py - Bộ nạp dữ liệu Video Tensor 12 kênh thời gian
Tương thích linh hoạt cả 2 tham số: voc_root và data_dir
"""
import os
import sys
import numpy as np
import pandas as pd
import tensorflow as tf

def decode_and_resize_img(file_path, channels=3, target_size=(640, 640)):
    img_raw = tf.io.read_file(file_path)
    img = tf.io.decode_jpeg(img_raw, channels=channels)
    img = tf.image.resize(img, target_size, method="bilinear")
    return tf.cast(img, tf.float32) / 255.0

def random_horizontal_flip(tensor_multi_ch, bbox, exist):
    do_flip = tf.random.uniform([]) > 0.5
    if do_flip:
        tensor_multi_ch = tf.image.flip_left_right(tensor_multi_ch)
        xmin, ymin, xmax, ymax = bbox[0], bbox[1], bbox[2], bbox[3]
        new_xmin = 1.0 - xmax
        new_xmax = 1.0 - xmin
        bbox = tf.stack([new_xmin, ymin, new_xmax, ymax])
    return tensor_multi_ch, bbox, exist

def build_preprocess_fn(voc_root=None, data_dir=None, target_size=(640, 640), modality="rgbt", is_training=True):
    if voc_root is None:
        voc_root = data_dir
    orig_w, orig_h = 1920.0, 1080.0

    def preprocess_sample(vis_tm1_rel, vis_t_rel, vis_tp1_rel,
                          ir_tm1_rel, ir_t_rel, ir_tp1_rel,
                          xmin, ymin, xmax, ymax, exist):
        p_vis_tm1 = tf.strings.join([voc_root, "/", vis_tm1_rel])
        p_vis_t   = tf.strings.join([voc_root, "/", vis_t_rel])
        p_vis_tp1 = tf.strings.join([voc_root, "/", vis_tp1_rel])

        img_vis_tm1 = decode_and_resize_img(p_vis_tm1, channels=3, target_size=target_size)
        img_vis_t   = decode_and_resize_img(p_vis_t,   channels=3, target_size=target_size)
        img_vis_tp1 = decode_and_resize_img(p_vis_tp1, channels=3, target_size=target_size)

        if modality.lower() == "rgbt":
            p_ir_tm1 = tf.strings.join([voc_root, "/", ir_tm1_rel])
            p_ir_t   = tf.strings.join([voc_root, "/", ir_t_rel])
            p_ir_tp1 = tf.strings.join([voc_root, "/", ir_tp1_rel])

            img_ir_tm1 = decode_and_resize_img(p_ir_tm1, channels=1, target_size=target_size)
            img_ir_t   = decode_and_resize_img(p_ir_t,   channels=1, target_size=target_size)
            img_ir_tp1 = decode_and_resize_img(p_ir_tp1, channels=1, target_size=target_size)

            tensor_input = tf.concat([
                img_vis_tm1, img_vis_t, img_vis_tp1,
                img_ir_tm1,  img_ir_t,  img_ir_tp1
            ], axis=-1)
        elif modality.lower() == "rgb_motion":
            diff_prev = tf.reduce_mean(tf.abs(img_vis_t - img_vis_tm1), axis=-1, keepdims=True)
            diff_next = tf.reduce_mean(tf.abs(img_vis_tp1 - img_vis_t), axis=-1, keepdims=True)
            tensor_input = tf.concat([img_vis_t, diff_prev, diff_next], axis=-1)
        else:
            tensor_input = tf.concat([img_vis_tm1, img_vis_t, img_vis_tp1], axis=-1)

        norm_xmin = tf.clip_by_value(tf.cast(xmin, tf.float32) / orig_w, 0.0, 1.0)
        norm_ymin = tf.clip_by_value(tf.cast(ymin, tf.float32) / orig_h, 0.0, 1.0)
        norm_xmax = tf.clip_by_value(tf.cast(xmax, tf.float32) / orig_w, 0.0, 1.0)
        norm_ymax = tf.clip_by_value(tf.cast(ymax, tf.float32) / orig_h, 0.0, 1.0)

        norm_bbox = tf.where(
            exist == 1,
            tf.stack([norm_xmin, norm_ymin, norm_xmax, norm_ymax]),
            tf.constant([0.0, 0.0, 0.0, 0.0], dtype=tf.float32)
        )

        if is_training:
            tensor_input, norm_bbox, exist = random_horizontal_flip(tensor_input, norm_bbox, exist)

        # ---------------------------------------------------------------------
        # Sinh nhãn Anchor-Free trên lưới 80x80 (Stride = 8)
        # ---------------------------------------------------------------------
        grid_h, grid_w = 80, 80
        xs = tf.cast(tf.range(grid_w), tf.float32)
        ys = tf.cast(tf.range(grid_h), tf.float32)
        grid_x, grid_y = tf.meshgrid(xs, ys)

        cur_xmin, cur_ymin = norm_bbox[0], norm_bbox[1]
        cur_xmax, cur_ymax = norm_bbox[2], norm_bbox[3]

        cx = (cur_xmin + cur_xmax) / 2.0
        cy = (cur_ymin + cur_ymax) / 2.0
        bw = cur_xmax - cur_xmin
        bh = cur_ymax - cur_ymin

        px = cx * float(grid_w)
        py = cy * float(grid_h)

        ix = tf.clip_by_value(tf.cast(tf.math.floor(px), tf.int32), 0, grid_w - 1)
        iy = tf.clip_by_value(tf.cast(tf.math.floor(py), tf.int32), 0, grid_h - 1)

        dx = px - tf.cast(ix, tf.float32)
        dy = py - tf.cast(iy, tf.float32)

        sigma = tf.maximum(1.0, tf.sqrt(tf.maximum(bw * float(grid_w) * bh * float(grid_h), 1e-4)) / 3.0)
        dist_sq = tf.square(grid_x - tf.cast(ix, tf.float32)) + tf.square(grid_y - tf.cast(iy, tf.float32))
        gaussian = tf.exp(-dist_sq / (2.0 * tf.square(sigma)))

        center_point_mask = tf.cast(
            tf.logical_and(tf.equal(tf.cast(grid_x, tf.int32), ix), tf.equal(tf.cast(grid_y, tf.int32), iy)),
            tf.float32
        )
        center_point_mask = tf.expand_dims(center_point_mask, axis=-1)
        center_point_mask = tf.where(exist == 1, center_point_mask, tf.zeros_like(center_point_mask))

        heatmap = tf.where(exist == 1, tf.expand_dims(gaussian, axis=-1), tf.zeros([grid_h, grid_w, 1], dtype=tf.float32))
        offset_map = tf.concat([center_point_mask * dx, center_point_mask * dy], axis=-1)
        size_map = tf.concat([center_point_mask * bw, center_point_mask * bh], axis=-1)

        labels = {
            "heatmap": heatmap,
            "offset": offset_map,
            "size": size_map,
            "mask": center_point_mask,
            "raw_bbox": tf.cast(norm_bbox, tf.float32),
            "class_output": tf.cast(exist, tf.float32),
            "bbox_output": tf.cast(norm_bbox, tf.float32)
        }
        return tensor_input, labels

    return preprocess_sample

def create_temporal_dataset(csv_path, voc_root=None, data_dir=None, batch_size=8,
                            target_size=(640, 640), modality="rgbt",
                            is_training=True, shuffle_buffer=1024):
    if voc_root is None:
        voc_root = data_dir
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"Không tìm thấy file CSV tại: {csv_path}")

    print(f">> Khởi tạo DataLoader từ: {csv_path}")
    df = pd.read_csv(csv_path)
    total_samples = len(df)
    print(f"   + Tổng số mẫu: {total_samples} | Modality: {modality.upper()} | Batch: {batch_size}")

    ds = tf.data.Dataset.from_tensor_slices((
        df["vis_tm1"].values.astype(str),
        df["vis_t"].values.astype(str),
        df["vis_tp1"].values.astype(str),
        df["ir_tm1"].values.astype(str) if "ir_tm1" in df.columns else df["vis_tm1"].values.astype(str),
        df["ir_t"].values.astype(str) if "ir_t" in df.columns else df["vis_t"].values.astype(str),
        df["ir_tp1"].values.astype(str) if "ir_tp1" in df.columns else df["vis_tp1"].values.astype(str),
        df["xmin"].values.astype(np.float32),
        df["ymin"].values.astype(np.float32),
        df["xmax"].values.astype(np.float32),
        df["ymax"].values.astype(np.float32),
        df["exist"].values.astype(np.int32)
    ))

    if is_training:
        ds = ds.shuffle(buffer_size=min(total_samples, shuffle_buffer), reshuffle_each_iteration=True)

    preprocess_fn = build_preprocess_fn(voc_root, target_size=target_size, modality=modality, is_training=is_training)
    ds = ds.map(preprocess_fn, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size, drop_remainder=is_training)
    ds = ds.prefetch(buffer_size=tf.data.AUTOTUNE)
    return ds, total_samples

create_tf_dataset = create_temporal_dataset


In [ ]:
%%writefile model_tf.py
# -*- coding: utf-8 -*-
"""
model_tf.py - ResNet50v2 + P2-FPN + Decoupled Spatial Soft-Argmax Head
"""
import tensorflow as tf
from tensorflow.keras import layers, Model

class CoordinateAttention(layers.Layer):
    def __init__(self, reduction=16, **kwargs):
        super().__init__(**kwargs)
        self.reduction = reduction

    def build(self, input_shape):
        channels = input_shape[-1]
        reduced = max(8, channels // self.reduction)
        self.conv_shared = layers.Conv2D(reduced, kernel_size=1, strides=1)
        self.bn = layers.BatchNormalization()
        self.act = layers.Activation("relu")
        self.conv_h = layers.Conv2D(channels, kernel_size=1, activation="sigmoid")
        self.conv_w = layers.Conv2D(channels, kernel_size=1, activation="sigmoid")
        super().build(input_shape)

    def call(self, x):
        h, w = tf.shape(x)[1], tf.shape(x)[2]
        x_h = tf.reduce_mean(x, axis=2, keepdims=True)
        x_w = tf.reduce_mean(x, axis=1, keepdims=True)
        x_w_perm = tf.transpose(x_w, perm=[0, 2, 1, 3])
        concat = tf.concat([x_h, x_w_perm], axis=1)

        y = self.act(self.bn(self.conv_shared(concat)))
        y_h = y[:, :h, :, :]
        y_w = tf.transpose(y[:, h:, :, :], perm=[0, 2, 1, 3])
        return x * self.conv_h(y_h) * self.conv_w(y_w)

def bottleneck_v2(x, filters, stride=1, projection=False, name="block"):
    x_pre = layers.Activation("relu", name=f"{name}_pre_relu")(
        layers.BatchNormalization(name=f"{name}_pre_bn")(x)
    )
    if projection:
        shortcut = layers.Conv2D(filters * 4, kernel_size=1, strides=stride, use_bias=False, name=f"{name}_proj")(x_pre)
    elif stride > 1:
        shortcut = layers.MaxPooling2D(pool_size=1, strides=stride, name=f"{name}_pool")(x)
    else:
        shortcut = x

    c1 = layers.Activation("relu", name=f"{name}_relu1")(
        layers.BatchNormalization(name=f"{name}_bn1")(
            layers.Conv2D(filters, kernel_size=1, strides=1, use_bias=False, name=f"{name}_conv1")(x_pre)
        )
    )
    c2 = layers.Activation("relu", name=f"{name}_relu2")(
        layers.BatchNormalization(name=f"{name}_bn2")(
            layers.Conv2D(filters, kernel_size=3, strides=stride, padding="same", use_bias=False, name=f"{name}_conv2")(c1)
        )
    )
    c3 = layers.Conv2D(filters * 4, kernel_size=1, strides=1, use_bias=False, name=f"{name}_conv3")(c2)
    return layers.Add(name=f"{name}_add")([shortcut, c3])

def build_custom_resnet50v2_backbone(input_tensor):
    x = layers.Conv2D(64, kernel_size=7, strides=2, padding="same", use_bias=False, name="stem_conv")(input_tensor)
    x = layers.MaxPooling2D(pool_size=3, strides=2, padding="same", name="stem_pool")(
        layers.Activation("relu", name="stem_relu")(layers.BatchNormalization(name="stem_bn")(x))
    )

    x = bottleneck_v2(x, 64, stride=1, projection=True, name="stage1_b1")
    x = bottleneck_v2(x, 64, stride=1, projection=False, name="stage1_b2")
    c2 = bottleneck_v2(x, 64, stride=1, projection=False, name="stage1_b3")

    x = bottleneck_v2(c2, 128, stride=2, projection=True, name="stage2_b1")
    x = bottleneck_v2(x, 128, stride=1, projection=False, name="stage2_b2")
    x = bottleneck_v2(x, 128, stride=1, projection=False, name="stage2_b3")
    c3 = bottleneck_v2(x, 128, stride=1, projection=False, name="stage2_b4")

    x = bottleneck_v2(c3, 256, stride=2, projection=True, name="stage3_b1")
    for i in range(2, 7):
        x = bottleneck_v2(x, 256, stride=1, projection=False, name=f"stage3_b{i}")
    c4 = x

    x = bottleneck_v2(c4, 512, stride=2, projection=True, name="stage4_b1")
    x = bottleneck_v2(x, 512, stride=1, projection=False, name="stage4_b2")
    c5 = layers.Activation("relu", name="post_relu")(
        layers.BatchNormalization(name="post_bn")(bottleneck_v2(x, 512, stride=1, projection=False, name="stage4_b3"))
    )
    return c2, c3, c4, c5

def build_p2_fpn(c2, c3, c4, c5, fpn_dim=128):
    lat_c5 = layers.Conv2D(fpn_dim, kernel_size=1, name="fpn_lat_c5")(c5)
    lat_c4 = layers.Conv2D(fpn_dim, kernel_size=1, name="fpn_lat_c4")(c4)
    lat_c3 = layers.Conv2D(fpn_dim, kernel_size=1, name="fpn_lat_c3")(c3)
    lat_c2 = layers.Conv2D(fpn_dim, kernel_size=1, name="fpn_lat_c2")(c2)

    p5 = lat_c5
    p4 = layers.Add(name="fpn_add_p4")([lat_c4, layers.UpSampling2D(size=2, name="fpn_up_p5")(p5)])
    p3 = layers.Add(name="fpn_add_p3")([lat_c3, layers.UpSampling2D(size=2, name="fpn_up_p4")(p4)])
    p2 = layers.Add(name="fpn_add_p2")([lat_c2, layers.UpSampling2D(size=2, name="fpn_up_p3")(p3)])

    p5 = layers.Conv2D(fpn_dim, kernel_size=3, padding="same", name="fpn_smooth_p5")(p5)
    p4 = layers.Conv2D(fpn_dim, kernel_size=3, padding="same", name="fpn_smooth_p4")(p4)
    p3 = layers.Conv2D(fpn_dim, kernel_size=3, padding="same", name="fpn_smooth_p3")(p3)
    p2 = layers.Conv2D(fpn_dim, kernel_size=3, padding="same", name="fpn_smooth_p2")(p2)

    p2 = CoordinateAttention(name="ca_p2")(p2)
    p3 = CoordinateAttention(name="ca_p3")(p3)
    return p2, p3, p4, p5

class AnchorFreeCenterHead(layers.Layer):
    """
    AnchorFreeCenterHead - Đầu dò tìm không dùng hộp neo (CenterNet-style):
    - Nhánh 1: Heatmap (80, 80, 1) - Dự đoán xác suất tâm drone tại từng điểm ảnh.
               Khởi tạo bias = -4.595 (pi = 0.01) để ổn định gradient khi train from scratch.
    - Nhánh 2: Offset  (80, 80, 2) - Dự đoán sai số lượng tử hóa (dx, dy).
    - Nhánh 3: Size    (80, 80, 2) - Dự đoán chiều rộng và chiều cao (w, h) chuẩn hóa [0, 1].
    """
    def __init__(self, fpn_dim=128, **kwargs):
        super().__init__(**kwargs)
        self.fpn_dim = fpn_dim

    def build(self, input_shape):
        # 1. Nhánh Heatmap
        self.hm_conv1 = layers.Conv2D(128, 3, padding="same", activation="relu", name="hm_conv1")
        self.hm_bn1 = layers.BatchNormalization(name="hm_bn1")
        self.hm_out = layers.Conv2D(
            1, 1, padding="same", activation="sigmoid",
            bias_initializer=tf.keras.initializers.Constant(-4.595),
            dtype="float32", name="heatmap"
        )

        # 2. Nhánh Offset
        self.off_conv1 = layers.Conv2D(64, 3, padding="same", activation="relu", name="off_conv1")
        self.off_bn1 = layers.BatchNormalization(name="off_bn1")
        self.off_out = layers.Conv2D(2, 1, padding="same", dtype="float32", name="offset")

        # 3. Nhánh Size (w, h) - Khử triệt để Dying ReLU bằng Sigmoid & Prior Bias (-2.66)
        self.size_conv1 = layers.Conv2D(64, 3, padding="same", activation="relu", name="size_conv1")
        self.size_bn1 = layers.BatchNormalization(name="size_bn1")
        self.size_out = layers.Conv2D(
            2, 1, padding="same", activation="sigmoid",
            bias_initializer=tf.keras.initializers.Constant(-2.66),
            dtype="float32", name="size"
        )

        super().build(input_shape)

    def call(self, x):
        # Nhánh Heatmap
        h_hm = self.hm_bn1(self.hm_conv1(x))
        pred_hm = self.hm_out(h_hm)

        # Nhánh Offset
        h_off = self.off_bn1(self.off_conv1(x))
        pred_off = self.off_out(h_off)

        # Nhánh Size
        h_size = self.size_bn1(self.size_conv1(x))
        pred_size = self.size_out(h_size)

        return {
            "heatmap": pred_hm,
            "offset": pred_off,
            "size": pred_size
        }

def decode_detections(heatmap, offset, size, default_w=0.065, default_h=0.070, min_size=0.015):
    """
    Giải mã Bounding Box cực đại địa phương (Local Peak) không cần NMS:
    - Tìm peak qua MaxPool2D 3x3
    - Trích xuất Top-1 peak cho bài toán Single Target Anti-UAV
    - Bổ cứu kích thước thông minh (Prior Fallback): Khi nhánh size bị triệt tiêu (dying relu, w, h ~ 0),
      tự động áp dụng kích thước Drone Prior trung bình của tập Anti-UAV (default_w=0.065, default_h=0.070)
    - Trả về: scores [B], bboxes [B, 4] dạng [xmin, ymin, xmax, ymax]
    """
    hmax = tf.nn.max_pool2d(heatmap, ksize=3, strides=1, padding="SAME")
    keep = tf.cast(tf.equal(heatmap, hmax), tf.float32)
    peak_heatmap = heatmap * keep

    B = tf.shape(heatmap)[0]
    H = tf.shape(heatmap)[1]
    W = tf.shape(heatmap)[2]

    flat_peaks = tf.reshape(peak_heatmap, [B, H * W])
    top_scores, top_indices = tf.math.top_k(flat_peaks, k=1)
    score = top_scores[:, 0]
    idx = top_indices[:, 0]

    grid_y = tf.cast(idx // W, tf.float32)
    grid_x = tf.cast(idx % W, tf.float32)

    flat_offset = tf.reshape(offset, [B, H * W, 2])
    flat_size = tf.reshape(size, [B, H * W, 2])

    batch_indices = tf.range(B, dtype=tf.int32)
    gather_idx = tf.stack([batch_indices, idx], axis=-1)

    peak_offset = tf.gather_nd(flat_offset, gather_idx)
    peak_size = tf.gather_nd(flat_size, gather_idx)

    cx = (grid_x + peak_offset[:, 0]) / tf.cast(W, tf.float32)
    cy = (grid_y + peak_offset[:, 1]) / tf.cast(H, tf.float32)
    bw = peak_size[:, 0]
    bh = peak_size[:, 1]

    # Bổ cứu kích thước thông minh: Nếu dự đoán quá nhỏ do Dying ReLU (< min_size), dùng Drone Prior
    bw = tf.where(bw < min_size, tf.constant(default_w, dtype=bw.dtype), bw)
    bh = tf.where(bh < min_size, tf.constant(default_h, dtype=bh.dtype), bh)

    xmin = tf.clip_by_value(cx - bw / 2.0, 0.0, 1.0)
    ymin = tf.clip_by_value(cy - bh / 2.0, 0.0, 1.0)
    xmax = tf.clip_by_value(cx + bw / 2.0, 0.0, 1.0)
    ymax = tf.clip_by_value(cy + bh / 2.0, 0.0, 1.0)

    bboxes = tf.stack([xmin, ymin, xmax, ymax], axis=-1)
    return score, bboxes

# Định danh tương thích ngược
DecoupledSpatialHead = AnchorFreeCenterHead
UAVSpatialDetectionHead = AnchorFreeCenterHead
SpatialDetectionHead = AnchorFreeCenterHead

def build_detection_model(input_shape=(640, 640, 12), fpn_dim=128, name="ResNet50v2_P2FPN"):
    inputs = layers.Input(shape=input_shape, name="temporal_input")
    c2, c3, c4, c5 = build_custom_resnet50v2_backbone(inputs)
    p2, p3, p4, p5 = build_p2_fpn(c2, c3, c4, c5, fpn_dim=fpn_dim)

    p2_down = layers.MaxPooling2D(pool_size=2, name="fuse_p2_down")(p2)
    p3_same = p3
    p4_up   = layers.UpSampling2D(size=2, name="fuse_p4_up")(p4)
    p5_up   = layers.UpSampling2D(size=4, name="fuse_p5_up")(p5)

    fused = layers.Concatenate(axis=-1, name="fuse_concat")([p2_down, p3_same, p4_up, p5_up])
    fused = layers.BatchNormalization(name="fuse_bn")(
        layers.Conv2D(fpn_dim, kernel_size=3, padding="same", activation="relu", name="fuse_conv")(fused)
    )

    outputs = AnchorFreeCenterHead(fpn_dim=fpn_dim, name="anchor_free_head")(fused)
    return Model(inputs=inputs, outputs=outputs, name=name)


In [ ]:
%%writefile evaluate_tf.py
# -*- coding: utf-8 -*-
"""
Phase 5: evaluate_tf.py - Đánh giá Định lượng Độc lập & Dự đoán Trực quan
Chỉ số: Precision, Recall, F1, mAP@0.5, mAP@0.75, COCO mAP@[0.5:0.95], NWD-mAP@0.5, Sai số tâm (px), FPS
"""
import os
import sys
import argparse
import numpy as np
import pandas as pd
import tensorflow as tf
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as patches

import dataset_tf
import model_tf

# Cấu hình Memory Growth để tránh lỗi OOM (ResourceExhaustedError) trên GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except Exception:
            pass

def compute_iou(b1, b2):
    x1 = np.maximum(b1[:, 0], b2[:, 0])
    y1 = np.maximum(b1[:, 1], b2[:, 1])
    x2 = np.minimum(b1[:, 2], b2[:, 2])
    y2 = np.minimum(b1[:, 3], b2[:, 3])
    inter = np.maximum(0.0, x2 - x1) * np.maximum(0.0, y2 - y1)
    a1 = np.maximum(0.0, b1[:, 2] - b1[:, 0]) * np.maximum(0.0, b1[:, 3] - b1[:, 1])
    a2 = np.maximum(0.0, b2[:, 2] - b2[:, 0]) * np.maximum(0.0, b2[:, 3] - b2[:, 1])
    union = a1 + a2 - inter
    return np.where(union > 1e-7, inter / union, 0.0)

def compute_nwd(b1, b2, c_norm=0.02):
    cx1, cy1 = (b1[:, 0] + b1[:, 2]) / 2.0, (b1[:, 1] + b1[:, 3]) / 2.0
    w1, h1   = np.maximum(b1[:, 2] - b1[:, 0], 1e-4), np.maximum(b1[:, 3] - b1[:, 1], 1e-4)
    cx2, cy2 = (b2[:, 0] + b2[:, 2]) / 2.0, (b2[:, 1] + b2[:, 3]) / 2.0
    w2, h2   = np.maximum(b2[:, 2] - b2[:, 0], 1e-4), np.maximum(b2[:, 3] - b2[:, 1], 1e-4)
    d2 = (cx1 - cx2)**2 + (cy1 - cy2)**2 + ((w1 - w2)**2 + (h1 - h2)**2) / 4.0
    return np.exp(-np.sqrt(np.maximum(d2, 1e-7)) / c_norm)

def compute_ap(y_true, y_scores, matches):
    total_pos = np.sum(y_true)
    if total_pos == 0:
        return 0.0, [0.0], [0.0]
    idx = np.argsort(-y_scores)
    tp = (y_true[idx] & matches[idx]).astype(float)
    fp = ((~y_true[idx]) | (~matches[idx])).astype(float)
    rec = np.cumsum(tp) / total_pos
    prec = np.cumsum(tp) / np.maximum(np.cumsum(tp) + np.cumsum(fp), 1e-7)
    mrec = np.concatenate(([0.0], rec, [1.0]))
    mpre = np.concatenate(([0.0], prec, [0.0]))
    for i in range(len(mpre) - 1, 0, -1):
        mpre[i - 1] = np.maximum(mpre[i - 1], mpre[i])
    inds = np.where(mrec[1:] != mrec[:-1])[0]
    return float(np.sum((mrec[inds + 1] - mrec[inds]) * mpre[inds + 1])), rec, prec

def decode_and_resize_eval(file_path, channels=3, target_size=(640, 640)):
    img_raw = tf.io.read_file(file_path)
    # ratio=2: Giải mã trực tiếp từ 1080p về 540p ở tầng libjpeg C++, tăng tốc 3-4 lần!
    img = tf.io.decode_jpeg(img_raw, channels=channels, ratio=2)
    img = tf.image.resize(img, target_size, method="bilinear")
    return tf.cast(img, tf.float32) / 255.0

def create_eval_dataset(csv_path, voc_root, batch_size=8, target_size=(640, 640), stride=1, max_samples=None, modality="rgb"):
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"Không tìm thấy file CSV tại: {csv_path}")

    df = pd.read_csv(csv_path)
    orig_total = len(df)

    if stride > 1:
        df = df.iloc[::stride].reset_index(drop=True)
    if max_samples is not None and max_samples > 0 and len(df) > max_samples:
        indices = np.linspace(0, len(df) - 1, max_samples, dtype=int)
        df = df.iloc[indices].reset_index(drop=True)

    total_samples = len(df)
    print(f"   + Tổng số mẫu gốc: {orig_total} | Đang lấy mẫu đánh giá: {total_samples} mẫu (Stride: {stride}, Modality: {modality.upper()}) | Batch: {batch_size}")

    orig_w, orig_h = 1920.0, 1080.0

    def preprocess_eval(vis_tm1_rel, vis_t_rel, vis_tp1_rel,
                        ir_tm1_rel, ir_t_rel, ir_tp1_rel,
                        xmin, ymin, xmax, ymax, exist):
        p_vis_tm1 = tf.strings.join([voc_root, "/", vis_tm1_rel])
        p_vis_t   = tf.strings.join([voc_root, "/", vis_t_rel])
        p_vis_tp1 = tf.strings.join([voc_root, "/", vis_tp1_rel])

        img_v_tm1 = decode_and_resize_eval(p_vis_tm1, 3, target_size)
        img_v_t   = decode_and_resize_eval(p_vis_t,   3, target_size)
        img_v_tp1 = decode_and_resize_eval(p_vis_tp1, 3, target_size)

        if modality.lower() == "rgbt":
            p_ir_tm1  = tf.strings.join([voc_root, "/", ir_tm1_rel])
            p_ir_t    = tf.strings.join([voc_root, "/", ir_t_rel])
            p_ir_tp1  = tf.strings.join([voc_root, "/", ir_tp1_rel])
            tensor_input = tf.concat([
                img_v_tm1, img_v_t, img_v_tp1,
                decode_and_resize_eval(p_ir_tm1, 1, target_size),
                decode_and_resize_eval(p_ir_t,   1, target_size),
                decode_and_resize_eval(p_ir_tp1, 1, target_size)
            ], axis=-1)
        elif modality.lower() == "rgb_motion":
            diff_prev = tf.reduce_mean(tf.abs(img_v_t - img_v_tm1), axis=-1, keepdims=True)
            diff_next = tf.reduce_mean(tf.abs(img_v_tp1 - img_v_t), axis=-1, keepdims=True)
            tensor_input = tf.concat([img_v_t, diff_prev, diff_next], axis=-1)
        else:
            tensor_input = tf.concat([img_v_tm1, img_v_t, img_v_tp1], axis=-1)

        norm_xmin = tf.clip_by_value(tf.cast(xmin, tf.float32) / orig_w, 0.0, 1.0)
        norm_ymin = tf.clip_by_value(tf.cast(ymin, tf.float32) / orig_h, 0.0, 1.0)
        norm_xmax = tf.clip_by_value(tf.cast(xmax, tf.float32) / orig_w, 0.0, 1.0)
        norm_ymax = tf.clip_by_value(tf.cast(ymax, tf.float32) / orig_h, 0.0, 1.0)

        norm_bbox = tf.where(
            exist == 1,
            tf.stack([norm_xmin, norm_ymin, norm_xmax, norm_ymax]),
            tf.constant([0.0, 0.0, 0.0, 0.0], dtype=tf.float32)
        )

        labels = {
            "class_output": tf.cast(exist, tf.float32),
            "bbox_output": tf.cast(norm_bbox, tf.float32)
        }
        return tensor_input, labels

    ds = tf.data.Dataset.from_tensor_slices((
        df["vis_tm1"].values.astype(str),
        df["vis_t"].values.astype(str),
        df["vis_tp1"].values.astype(str),
        df["ir_tm1"].values.astype(str) if "ir_tm1" in df else df["vis_tm1"].values.astype(str),
        df["ir_t"].values.astype(str) if "ir_t" in df else df["vis_t"].values.astype(str),
        df["ir_tp1"].values.astype(str) if "ir_tp1" in df else df["vis_tp1"].values.astype(str),
        df["xmin"].values.astype(np.float32),
        df["ymin"].values.astype(np.float32),
        df["xmax"].values.astype(np.float32),
        df["ymax"].values.astype(np.float32),
        df["exist"].values.astype(np.int32)
    ))

    ds = ds.map(preprocess_eval, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size, drop_remainder=False)
    ds = ds.prefetch(buffer_size=tf.data.AUTOTUNE)
    return ds, total_samples

def evaluate(voc_root, csv_path, checkpoint_path, output_dir="/kaggle/working/report_figures", batch_size=8, conf_thresh=0.40, stride=5, max_samples=3000, modality="rgb"):
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs("/kaggle/working/checkpoints", exist_ok=True)
    print("=" * 70)
    print(f" BẮT ĐẦU ĐÁNH GIÁ ĐỊNH LƯỢNG MÔ HÌNH ANCHOR-FREE (MODALITY: {modality.upper()})")
    print(f" Weights     : {checkpoint_path}")
    print(f" Test CSV    : {csv_path}")
    print(f" Batch Size  : {batch_size}")
    print(f" Conf Thresh : {conf_thresh}")
    print(f" Stride/Skip : {stride}")
    print(f" Max Samples : {max_samples}")
    print("=" * 70)

    if modality.lower() == "rgbt":
        in_channels = 12
    elif modality.lower() == "rgb_motion":
        in_channels = 5
    else:
        in_channels = 9

    model = model_tf.build_detection_model(input_shape=(640, 640, in_channels), fpn_dim=128)
    model.load_weights(checkpoint_path)
    print(">> Đã nạp thành công mô hình Anchor-Free!")

    ds, total_samples = create_eval_dataset(
        csv_path=csv_path, voc_root=voc_root, batch_size=batch_size, stride=stride, max_samples=max_samples, modality=modality
    )

    all_gt_cls, all_gt_box, all_pr_cls, all_pr_box = [], [], [], []
    import time
    start = time.time()

    @tf.function
    def infer_step(batch_imgs):
        preds = model(batch_imgs, training=False)
        scores, pr_bboxes = model_tf.decode_detections(preds["heatmap"], preds["offset"], preds["size"])
        return scores, pr_bboxes

    for imgs, lbls in tqdm(ds, desc="Đang đánh giá tập Dữ liệu"):
        scores, pr_bboxes = infer_step(imgs)
        all_gt_cls.extend(lbls["class_output"].numpy())
        all_gt_box.extend(lbls["bbox_output"].numpy())
        all_pr_cls.extend(scores.numpy())
        all_pr_box.extend(pr_bboxes.numpy())

    infer_time = time.time() - start
    fps = total_samples / max(infer_time, 1e-4)

    all_gt_cls = np.array(all_gt_cls)[:total_samples]
    all_gt_box = np.array(all_gt_box)[:total_samples]
    all_pr_cls = np.array(all_pr_cls)[:total_samples].flatten()
    all_pr_box = np.array(all_pr_box)[:total_samples]

    ious = compute_iou(all_pr_box, all_gt_box)
    nwds = compute_nwd(all_pr_box, all_gt_box)

    gt_pos = all_gt_cls >= 0.5
    pr_pos = all_pr_cls >= conf_thresh

    tp = np.sum(gt_pos & pr_pos)
    fp = np.sum((~gt_pos) & pr_pos)
    fn = np.sum(gt_pos & (~pr_pos))

    prec = tp / max(tp + fp, 1)
    rec  = tp / max(tp + fn, 1)
    f1   = 2 * prec * rec / max(prec + rec, 1e-7)

    ap_50, r_curve, p_curve = compute_ap(gt_pos, all_pr_cls, ious >= 0.5)
    ap_75, _, _ = compute_ap(gt_pos, all_pr_cls, ious >= 0.75)
    nwd_ap, _, _ = compute_ap(gt_pos, all_pr_cls, nwds >= 0.5)

    pos_mask = gt_pos & pr_pos
    if np.sum(pos_mask) > 0:
        cx_gt = (all_gt_box[:, 0] + all_gt_box[:, 2]) / 2.0 * 640.0
        cy_gt = (all_gt_box[:, 1] + all_gt_box[:, 3]) / 2.0 * 640.0
        cx_pr = (all_pr_box[:, 0] + all_pr_box[:, 2]) / 2.0 * 640.0
        cy_pr = (all_pr_box[:, 1] + all_pr_box[:, 3]) / 2.0 * 640.0
        center_err = float(np.mean(np.sqrt((cx_gt - cx_pr)**2 + (cy_gt - cy_pr)**2)[pos_mask]))
    else:
        center_err = 0.0

    print("\n" + "=" * 70)
    print(" KẾT QUẢ ĐÁNH GIÁ CHÍNH THỨC CHO BÁO CÁO")
    print("=" * 70)
    print(f" - Tổng số mẫu đánh giá : {total_samples} frames tiêu biểu")
    print(f" - Precision (Conf>={conf_thresh}): {prec * 100:.2f}%")
    print(f" - Recall (Conf>={conf_thresh})   : {rec * 100:.2f}%")
    print(f" - F1-Score           : {f1 * 100:.2f}%")
    print(f" - mAP@0.5 (IoU)      : {ap_50 * 100:.2f}%")
    print(f" - mAP@0.75 (IoU)     : {ap_75 * 100:.2f}%")
    print(f" - NWD-mAP@0.5 (NWD)  : {nwd_ap * 100:.2f}% (Metric chuẩn cho Drone nhỏ)")
    print(f" - Center Error (px)  : {center_err:.2f} pixels")
    print(f" - Tốc độ Suy Luận    : {fps:.1f} FPS trên GPU")
    print(f" - Thời gian thực hiện: {infer_time/60.0:.2f} phút")
    print("=" * 70)

    plt.figure(figsize=(8, 6), dpi=300)
    plt.plot(r_curve, p_curve, "b-", linewidth=2.5, label=f"Proposed Model (mAP@0.5 = {ap_50*100:.1f}%)")
    plt.xlabel("Recall", fontsize=12, fontweight="bold")
    plt.ylabel("Precision", fontsize=12, fontweight="bold")
    plt.title("Đường Cong Precision-Recall (Anchor-Free CenterNet + P2-FPN)", fontsize=13, fontweight="bold")
    plt.grid(True, linestyle=":", alpha=0.6)
    plt.legend(fontsize=11)
    
    pr_path = os.path.join(output_dir, "pr_curve.png")
    plt.savefig(pr_path, bbox_inches="tight")
    plt.savefig("/kaggle/working/checkpoints/pr_curve.png", bbox_inches="tight")
    plt.close()
    print(f">> Đã lưu PR Curve tại: {pr_path}")

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--voc_root", "--data_dir", dest="voc_root", type=str, required=True)
    parser.add_argument("--csv_path", type=str, required=True)
    parser.add_argument("--checkpoint", type=str, required=True)
    parser.add_argument("--conf_thresh", type=float, default=0.40)
    parser.add_argument("--batch_size", type=int, default=8)
    parser.add_argument("--stride", type=int, default=5, help="Bước nhảy lấy mẫu khung hình (mặc định 5, giảm thời gian chạy từ 6h xuống vài phút)")
    parser.add_argument("--max_samples", type=int, default=3000, help="Số lượng mẫu tối đa đánh giá (mặc định 3000 mẫu phân bổ đều)")
    parser.add_argument("--modality", type=str, default="rgb", choices=["rgb", "rgbt", "rgb_motion"], help="Chế độ đầu vào: rgb (9 kênh), rgbt (12 kênh), rgb_motion (5 kênh)")
    args = parser.parse_args()
    evaluate(
        args.voc_root, args.csv_path, args.checkpoint,
        batch_size=args.batch_size, conf_thresh=args.conf_thresh,
        stride=args.stride, max_samples=args.max_samples,
        modality=args.modality
    )


In [ ]:
%%writefile infer_video.py
# -*- coding: utf-8 -*-
"""
infer_video.py - Tạo Video Demo Tác Chiến với Giao diện HUD và Trajectory EMA Filter
Tự động xuất chuẩn H.264 (avc1) tương thích 100% Web Player và Báo cáo
"""
import os
import sys
import glob
import time
import argparse
import xml.etree.ElementTree as ET
import numpy as np
import cv2
import tensorflow as tf
from tqdm import tqdm

import model_tf

class TrajectoryEMAFilter:
    def __init__(self, alpha=0.6, max_missing=15):
        self.alpha = alpha
        self.max_missing = max_missing
        self.smooth_box = None
        self.velocity = np.array([0.0, 0.0])
        self.missing_count = 0
        self.is_active = False

    def update(self, detected_box, is_detected):
        if is_detected and detected_box is not None:
            cur_box = np.array(detected_box, dtype=np.float32)
            cur_cx = (cur_box[0] + cur_box[2]) / 2.0
            cur_cy = (cur_box[1] + cur_box[3]) / 2.0

            if self.smooth_box is None:
                self.smooth_box = cur_box
                self.velocity = np.array([0.0, 0.0])
            else:
                prev_cx = (self.smooth_box[0] + self.smooth_box[2]) / 2.0
                prev_cy = (self.smooth_box[1] + self.smooth_box[3]) / 2.0
                self.velocity = 0.6 * self.velocity + 0.4 * np.array([cur_cx - prev_cx, cur_cy - prev_cy])
                self.smooth_box = self.alpha * cur_box + (1.0 - self.alpha) * self.smooth_box

            self.missing_count = 0
            self.is_active = True
            return self.smooth_box.copy(), "DETECTED", float(np.linalg.norm(self.velocity))
        else:
            if self.is_active and self.missing_count < self.max_missing:
                self.missing_count += 1
                self.smooth_box[0] += self.velocity[0]
                self.smooth_box[2] += self.velocity[0]
                self.smooth_box[1] += self.velocity[1]
                self.smooth_box[3] += self.velocity[1]
                return self.smooth_box.copy(), "OCCLUDED", float(np.linalg.norm(self.velocity))
            else:
                self.is_active = False
                self.smooth_box = None
                return None, "LOST", 0.0

def draw_hud_reticle(img, bbox, conf=1.0, speed=0.0, state="DETECTED"):
    x1, y1, x2, y2 = [int(v) for v in bbox]
    w, h = x2 - x1, y2 - y1
    cx, cy = (x1 + x2) // 2, (y1 + y2) // 2

    color = (0, 255, 0) if state == "DETECTED" else (0, 215, 255)
    line_len = max(8, min(w, h) // 3)
    t = 2

    # 4 góc định vị
    cv2.line(img, (x1, y1), (x1 + line_len, y1), color, t)
    cv2.line(img, (x1, y1), (x1, y1 + line_len), color, t)
    cv2.line(img, (x2, y1), (x2 - line_len, y1), color, t)
    cv2.line(img, (x2, y1), (x2, y1 + line_len), color, t)
    cv2.line(img, (x1, y2), (x1 + line_len, y2), color, t)
    cv2.line(img, (x1, y2), (x1, y2 - line_len), color, t)
    cv2.line(img, (x2, y2), (x2 - line_len, y2), color, t)
    cv2.line(img, (x2, y2), (x2, y2 - line_len), color, t)

    cv2.rectangle(img, (x1, y1), (x2, y2), color, 1)
    cv2.circle(img, (cx, cy), 2, color, -1)

    info_str = f"DRONE {conf*100:.1f}% | {w}x{h}px | v={speed:.1f}px/f"
    (fw, fh), _ = cv2.getTextSize(info_str, cv2.FONT_HERSHEY_SIMPLEX, 0.45, 1)
    cv2.rectangle(img, (x1, max(0, y1 - fh - 8)), (x1 + fw + 8, max(0, y1 - fh - 8) + fh + 6), (20, 20, 20), -1)
    cv2.rectangle(img, (x1, max(0, y1 - fh - 8)), (x1 + fw + 8, max(0, y1 - fh - 8) + fh + 6), color, 1)
    cv2.putText(img, info_str, (x1 + 4, max(0, y1 - 2)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1, cv2.LINE_AA)

def render_sequence_video(voc_root, checkpoint_path, sequence_id, output_path, max_frames=200, conf_thresh=0.40, modality="rgb", scale_label=""):
    os.makedirs(os.path.dirname(os.path.abspath(output_path)), exist_ok=True)
    if modality.lower() == "rgbt":
        in_channels = 12
    elif modality.lower() == "rgb_motion":
        in_channels = 5
    else:
        in_channels = 9

    try:
        model = tf.keras.models.load_model(checkpoint_path, custom_objects={"CoordinateAttention": model_tf.CoordinateAttention, "AnchorFreeCenterHead": model_tf.AnchorFreeCenterHead}, compile=False)
    except Exception:
        model = model_tf.build_detection_model(input_shape=(640, 640, in_channels), fpn_dim=128)
        model.load_weights(checkpoint_path)

    jpeg_dir = os.path.join(voc_root, "JPEGImages")
    vis_files = sorted(glob.glob(os.path.join(jpeg_dir, f"{sequence_id}_vis_*.jpg")))
    if len(vis_files) == 0:
        raise FileNotFoundError(f"Không tìm thấy ảnh của chuỗi: {sequence_id}")

    total_frames = min(len(vis_files), max_frames)
    vis_files = vis_files[:total_frames]

    out_w, out_h = 960, 540
    canvas_w, canvas_h = out_w, out_h + 60
    
    raw_avi = output_path.replace(".mp4", "_raw.mp4")
    writer = cv2.VideoWriter(raw_avi, cv2.VideoWriter_fourcc(*'mp4v'), 20.0, (canvas_w, canvas_h))
    ema_filter = TrajectoryEMAFilter(alpha=0.6, max_missing=15)

    def load_norm_rgb(path, is_gray=False):
        raw = cv2.imread(path)
        if raw is None:
            return np.zeros((640, 640, 1 if is_gray else 3), dtype=np.float32)
        im = cv2.resize(raw, (640, 640))
        if is_gray:
            return cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)[:, :, np.newaxis].astype(np.float32) / 255.0
        return cv2.cvtColor(im, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0

    print(f"\n>> Đang Render Video: {sequence_id} ({total_frames} frames, Modality: {modality.upper()})...")
    for i in tqdm(range(total_frames)):
        path_v_t = vis_files[i]
        path_v_tm1 = vis_files[max(0, i - 1)]
        path_v_tp1 = vis_files[min(total_frames - 1, i + 1)]

        raw_v = cv2.imread(path_v_t)
        if raw_v is None:
            continue
        orig_h, orig_w = raw_v.shape[:2]

        t_in = np.concatenate([
            load_norm_rgb(path_v_tm1, False), load_norm_rgb(path_v_t, False), load_norm_rgb(path_v_tp1, False)
        ], axis=-1)[np.newaxis, ...]

        preds = model(t_in, training=False)
        scores, bboxes = model_tf.decode_detections(preds["heatmap"], preds["offset"], preds["size"])
        conf = float(scores[0].numpy())
        box_n = bboxes[0].numpy()
        abs_box = [box_n[0]*orig_w, box_n[1]*orig_h, box_n[2]*orig_w, box_n[3]*orig_h]

        final_box, state, spd = ema_filter.update(abs_box, conf >= conf_thresh)

        # Chi hien thi duy nhat 1 camera Visible RGB
        disp_v = cv2.resize(raw_v, (out_w, out_h))
        sx, sy = out_w / orig_w, out_h / orig_h

        if final_box is not None:
            scaled_box = [final_box[0]*sx, final_box[1]*sy, final_box[2]*sx, final_box[3]*sy]
            draw_hud_reticle(disp_v, scaled_box, conf, spd, state)

        # Header HUD voi tieu de chi ghi "ANTI-UAV RGB"
        hud = np.zeros((60, canvas_w, 3), dtype=np.uint8)
        hud[:, :] = (20, 25, 30)
        cv2.putText(hud, "ANTI-UAV RGB", (20, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 255, 255), 2)
        sub_info = f"Sequence: {sequence_id} | Frame {i+1:03d}/{total_frames:03d}"
        cv2.putText(hud, sub_info, (20, 48), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (180, 180, 180), 1)
        stat_color = (0, 255, 0) if state == "DETECTED" else ((0, 215, 255) if state == "OCCLUDED" else (100, 100, 100))
        cv2.putText(hud, f"STATUS: [{state}]", (canvas_w - 280, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.65, stat_color, 2)

        combined = np.vstack([hud, disp_v])
        writer.write(combined)

    writer.release()
    
    # Chuyển đổi mã hóa sang H.264 bằng ffmpeg để phát mượt mà trên HTML5 Web
    cmd = f'ffmpeg -y -i "{raw_avi}" -vcodec libx264 -pix_fmt yuv420p -crf 23 "{output_path}" -loglevel error'
    ret = os.system(cmd)
    if ret == 0 and os.path.exists(output_path):
        try: os.remove(raw_avi)
        except Exception: pass
        print(f">> [H.264] Đã nén chuẩn Web H.264 thành công: {output_path}")
    else:
        if os.path.exists(raw_avi):
            if os.path.exists(output_path): os.remove(output_path)
            os.rename(raw_avi, output_path)
        print(f"ĐÃ RENDER XONG: {output_path}")

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--voc_root", "--data_dir", dest="voc_root", type=str, required=True)
    parser.add_argument("--checkpoint", type=str, required=True)
    parser.add_argument("--seq_id", type=str, required=True)
    parser.add_argument("--output", type=str, required=True)
    parser.add_argument("--max_frames", type=int, default=200)
    parser.add_argument("--conf_thresh", type=float, default=0.40)
    parser.add_argument("--modality", type=str, default="rgb", choices=["rgb", "rgbt", "rgb_motion"], help="Chế độ đầu vào: rgb (9 kênh), rgbt (12 kênh), rgb_motion (5 kênh)")
    parser.add_argument("--scale_label", type=str, default="", help="Nhãn phân khúc kích thước drone (To, Trung bình, Nhỏ)")
    args = parser.parse_args()

    render_sequence_video(
        args.voc_root, args.checkpoint, args.seq_id, args.output,
        args.max_frames, conf_thresh=args.conf_thresh, modality=args.modality, scale_label=args.scale_label
    )

In [ ]:
%%writefile generate_report_figures.py
# -*- coding: utf-8 -*-
"""
generate_report_figures.py - Tự động tạo các hình ảnh khoa học chất lượng cao (300 DPI) cho Báo cáo / Bài báo
"""
import os
import glob
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

def create_report_figures(voc_root, csv_path, output_dir="/kaggle/working/report_figures"):
    os.makedirs(output_dir, exist_ok=True)
    df = pd.read_csv(csv_path)
    pos_df = df[df["exist"] == 1].reset_index(drop=True)
    print(f">> Tổng số mẫu có Drone để trích xuất: {len(pos_df)}")

    # FIGURE 1: Triplet Đa thời gian RGB kèm Kính lúp phóng to
    sample = pos_df.iloc[min(42, len(pos_df) - 1)]
    
    p_v_tm1 = os.path.join(voc_root, sample["vis_tm1"])
    p_v_t   = os.path.join(voc_root, sample["vis_t"])
    p_v_tp1 = os.path.join(voc_root, sample["vis_tp1"])

    img_v_tm1 = cv2.cvtColor(cv2.imread(p_v_tm1), cv2.COLOR_BGR2RGB)
    img_v_t   = cv2.cvtColor(cv2.imread(p_v_t),   cv2.COLOR_BGR2RGB)
    img_v_tp1 = cv2.cvtColor(cv2.imread(p_v_tp1), cv2.COLOR_BGR2RGB)

    diff_prev = np.mean(np.abs(img_v_t.astype(float) - img_v_tm1.astype(float)), axis=-1)
    diff_next = np.mean(np.abs(img_v_tp1.astype(float) - img_v_t.astype(float)), axis=-1)

    fig, axes = plt.subplots(2, 3, figsize=(16, 9), dpi=300)
    axes[0, 0].imshow(img_v_tm1); axes[0, 0].set_title("(a) Visible Frame (t-1)", fontsize=13, fontweight="bold")
    axes[0, 1].imshow(img_v_t);   axes[0, 1].set_title("(b) Visible Frame (t) [Target Frame]", fontsize=13, fontweight="bold", color="darkred")
    axes[0, 2].imshow(img_v_tp1); axes[0, 2].set_title("(c) Visible Frame (t+1)", fontsize=13, fontweight="bold")

    axes[1, 0].imshow(diff_prev, cmap="hot"); axes[1, 0].set_title("(d) Temporal Motion Diff |t - (t-1)|", fontsize=13, fontweight="bold")
    axes[1, 1].imshow(img_v_t);               axes[1, 1].set_title("(e) Drone Ground Truth Zoom-in", fontsize=13, fontweight="bold", color="darkred")
    axes[1, 2].imshow(diff_next, cmap="hot"); axes[1, 2].set_title("(f) Temporal Motion Diff |(t+1) - t|", fontsize=13, fontweight="bold")

    for ax in axes.ravel(): ax.axis("off")

    xmin, ymin, xmax, ymax = sample["xmin"], sample["ymin"], sample["xmax"], sample["ymax"]
    bw, bh = xmax - xmin, ymax - ymin

    for target_ax in [axes[0, 1], axes[1, 1]]:
        rect = patches.Rectangle((xmin, ymin), bw, bh, linewidth=2.5, edgecolor="#00FF00", facecolor="none")
        target_ax.add_patch(rect)
        target_ax.text(xmin, max(0, ymin - 12), f"DRONE ({bw:.0f}x{bh:.0f}px)", color="white", fontsize=10, fontweight="bold",
                       bbox=dict(facecolor="#00AA00", edgecolor="none", pad=2))

        ax_ins = inset_axes(target_ax, width="28%", height="28%", loc="lower left", borderpad=1)
        pad = max(bw, bh) * 1.5
        cx, cy = (xmin + xmax) / 2.0, (ymin + ymax) / 2.0
        z_xmin, z_xmax = max(0, int(cx - pad)), min(1920, int(cx + pad))
        z_ymin, z_ymax = max(0, int(cy - pad)), min(1080, int(cy + pad))

        ax_ins.imshow(img_v_t[z_ymin:z_ymax, z_xmin:z_xmax])
        ax_ins.add_patch(patches.Rectangle((xmin - z_xmin, ymin - z_ymin), bw, bh, linewidth=2, edgecolor="#00FF00", facecolor="none"))
        ax_ins.set_xticks([]); ax_ins.set_yticks([])
        for spine in ax_ins.spines.values():
            spine.set_edgecolor("#00FF00")
            spine.set_linewidth(2.5)

    plt.tight_layout()
    fig1_path = os.path.join(output_dir, "fig1_rgbt_temporal_triplet.png")
    plt.savefig(fig1_path, bbox_inches="tight")
    plt.close()
    print(f">> [1/2] Đã xuất Fig 1 (Triplet Đa thời gian RGB): {fig1_path}")

    # FIGURE 2: Phân bố kích thước Small Object MS-COCO
    widths = pos_df["xmax"] - pos_df["xmin"]
    heights = pos_df["ymax"] - pos_df["ymin"]
    areas_px = widths * heights

    tiny_mask = (widths < 32) & (heights < 32)
    tiny_pct = (np.sum(tiny_mask) / len(pos_df)) * 100.0

    fig, ax = plt.subplots(1, 2, figsize=(14, 5), dpi=300)
    ax[0].hist(widths, bins=35, color="#1f77b4", alpha=0.8, edgecolor="black", label="Chiều rộng (Width)")
    ax[0].hist(heights, bins=35, color="#ff7f0e", alpha=0.7, edgecolor="black", label="Chiều cao (Height)")
    ax[0].axvline(32, color="red", linestyle="--", linewidth=2.5, label="Ngưỡng Tiny Object (< 32px)")
    ax[0].set_title(f"Phân Bố Kích Thước Drone ({tiny_pct:.1f}% là Tiny Drone < 32px)", fontsize=12, fontweight="bold")
    ax[0].set_xlabel("Kích thước (Pixels)", fontsize=11, fontweight="bold")
    ax[0].set_ylabel("Số lượng mẫu (Frames)", fontsize=11, fontweight="bold")
    ax[0].legend(fontsize=10)
    ax[0].grid(True, linestyle=":", alpha=0.6)

    small_count = np.sum(areas_px < (32 * 32))
    medium_count = np.sum((areas_px >= (32 * 32)) & (areas_px < (96 * 96)))
    large_count = np.sum(areas_px >= (96 * 96))

    labels = [f"Tiny / Small (<32² px)\n{small_count/len(pos_df)*100:.1f}%",
              f"Medium (32² - 96² px)\n{medium_count/len(pos_df)*100:.1f}%",
              f"Large (>96² px)\n{large_count/len(pos_df)*100:.1f}%"]
    colors = ["#ff4d4d", "#ffaa00", "#3399ff"]
    ax[1].pie([small_count, medium_count, large_count], labels=labels, colors=colors, startangle=140,
              explode=(0.08, 0.05, 0.1), autopct="%1.1f%%", textprops={'fontsize': 10, 'weight': 'bold'})
    ax[1].set_title("Tỷ Lệ Phân Hạng Vật Thể Chuẩn MS-COCO", fontsize=12, fontweight="bold")

    plt.tight_layout()
    fig2_path = os.path.join(output_dir, "fig2_drone_scale_distribution.png")
    plt.savefig(fig2_path, bbox_inches="tight")
    plt.close()
    print(f">> [2/2] Đã xuất Fig 2 (Phân bố kích thước Small Object): {fig2_path}")

if __name__ == "__main__":
    import glob
    voc = "/kaggle/input/datasets/namnguyen171006/anti-uav-rgb/VOC_AntiUAV_RGB_stride3/VOC_AntiUAV_RGB"
    if not os.path.exists(voc):
        f = glob.glob("/kaggle/input/**/VOC_AntiUAV_RGB", recursive=True)
        if f: voc = f[0]
    csv = os.path.join(voc, "train_triplets.csv")
    if not os.path.exists(csv):
        f = glob.glob("/kaggle/input/**/train_triplets.csv", recursive=True)
        if f: csv = f[0]
    create_report_figures(voc_root=voc, csv_path=csv)


---
## PHẦN 1: VẼ BIỂU ĐỒ HỘI TỤ LOSS & XUẤT HÌNH ẢNH KHOA HỌC BÁO CÁO (300 DPI)
* **Biểu đồ Loss**: Thể hiện tiến trình giảm ngoạn mục của Total Loss và Heatmap Gaussian Focal Loss qua 9 Epochs.
* **Fig 1**: Mẫu Triplet Đa thời gian RGB kèm ô phóng to Inset Zoom-in chi tiết Drone.
* **Fig 2**: Biểu đồ phân bố tỷ lệ vật thể kích thước siêu nhỏ chuẩn MS-COCO.


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

history_csv = "/kaggle/working/checkpoints/training_history.csv"
if os.path.exists(history_csv):
    df = pd.read_csv(history_csv)

    plt.figure(figsize=(14, 5), dpi=300)

    # Subplot 1: Total Loss
    plt.subplot(1, 2, 1)
    plt.plot(df["epoch"], df["train_loss"], "b-o", linewidth=2.2, label="Train Loss")
    plt.plot(df["epoch"], df["val_loss"], "r--s", linewidth=2.2, label="Val Loss")
    best_ep = df.loc[df["val_loss"].idxmin(), "epoch"]
    best_vl = df["val_loss"].min()
    plt.scatter([best_ep], [best_vl], color="darkred", s=100, zorder=5, label=f"Best Val ({best_vl:.3f})")
    plt.title("Tiến trình Hội tụ Tổng Mất mát (Total Loss)", fontsize=12, fontweight="bold")
    plt.xlabel("Epoch", fontsize=11, fontweight="bold")
    plt.ylabel("Loss", fontsize=11, fontweight="bold")
    plt.grid(True, linestyle=":", alpha=0.6)
    plt.legend(fontsize=10)

    # Subplot 2: Heatmap Focal Loss & Size Loss
    plt.subplot(1, 2, 2)
    if "train_hm_loss" in df.columns:
        plt.plot(df["epoch"], df["train_hm_loss"], "g-^", linewidth=2, label="Train Heatmap (Gaussian Focal)")
        plt.plot(df["epoch"], df["val_hm_loss"], "m--d", linewidth=2, label="Val Heatmap (Gaussian Focal)")
        plt.plot(df["epoch"], df["train_size_loss"], "c:", linewidth=2, label="Train Size & Offset Loss")
        plt.plot(df["epoch"], df["val_size_loss"], "y-.", linewidth=2, label="Val Size & Offset Loss")
        plt.title("Chi tiết Heatmap Focal Loss & Size/Offset Loss", fontsize=12, fontweight="bold")
    elif "train_nwd_loss" in df.columns:
        plt.plot(df["epoch"], df["train_nwd_loss"], "g-^", linewidth=2, label="Train NWD (BBox)")
        plt.plot(df["epoch"], df["val_nwd_loss"], "m--d", linewidth=2, label="Val NWD (BBox)")
        plt.plot(df["epoch"], df["train_cls_loss"], "c:", linewidth=2, label="Train Focal (Cls)")
        plt.title("Chi tiết NWD BBox Loss & Focal Cls Loss", fontsize=12, fontweight="bold")
    plt.xlabel("Epoch", fontsize=11, fontweight="bold")
    plt.ylabel("Component Loss", fontsize=11, fontweight="bold")
    plt.grid(True, linestyle=":", alpha=0.6)
    plt.legend(fontsize=10)

    plt.tight_layout()
    loss_fig_path = "/kaggle/working/report_figures/loss_curve.png"
    plt.savefig(loss_fig_path, dpi=300)
    plt.savefig("/kaggle/working/checkpoints/loss_curve.png", dpi=300)
    plt.savefig("/kaggle/working/loss_curve.png", dpi=300)
    plt.show()
    print(f">> [x] ĐÃ LƯU BIỂU ĐỒ LOSS CHUẨN 300 DPI TẠI: {loss_fig_path}")
    print(f">> [x] Điểm tối ưu nhất đạt tại Epoch {int(best_ep)} với Val Loss = {best_vl:.4f}")
else:
    print(f"[CẢNH BÁO] Chưa tìm thấy file lịch sử tại: {history_csv}")


In [ ]:
# Chạy tạo 2 hình ảnh khoa học Fig 1 & Fig 2
import os
os.system("python generate_report_figures.py")


---
## PHẦN 2: ĐÁNH GIÁ ĐỊNH LƯỢNG ĐỘC LẬP (QUANTITATIVE EVALUATION)
* Sử dụng bộ giải mã **`decode_detections` (Local Peak MaxPool 3x3)** không cần NMS.
* Sử dụng **`batch_size = 8`** kết hợp **`@tf.function`** để tối ưu hóa bộ nhớ VRAM, đảm bảo không gặp lỗi OOM trên GPU Tesla P100 (16GB).
* Xuất các chỉ số chuẩn khoa học: **Precision, Recall, F1-Score, mAP@0.5, mAP@0.75, NWD-mAP@0.5, Sai số tâm (pixels), FPS**, và vẽ đường cong Precision-Recall Curve.

In [ ]:
# Chạy đánh giá định lượng độc lập trên tập Test RGB (3000 mẫu đại diện, chỉ mất ~5-8 phút!)
import os
cmd = (
    f'python evaluate_tf.py '
    f'--voc_root "{VOC_ROOT}" '
    f'--csv_path "{EVAL_CSV}" '
    f'--checkpoint "{DEST_CKPT}" '
    f'--modality rgb '
    f'--conf_thresh 0.40 '
    f'--batch_size 8 '
    f'--stride 5 '
    f'--max_samples 3000'
)
print(">> Đang thực thi đánh giá định lượng...")
os.system(cmd)


---
## PHẦN 3: DỰ ĐOÁN TRỰC QUAN TRÊN ẢNH TĨNH (STATIC VISUAL PREDICTIONS)
* Trích xuất ngẫu nhiên các mẫu kiểm thử chứa Drone từ tập dữ liệu.
* So sánh đối chiếu Ground Truth (khung xanh lá viền đứt) với Dự đoán của mô hình Anchor-Free (khung đỏ kèm độ tin cậy Conf %).
* Xuất file ảnh phân giải cao 300 DPI (`fig_visual_test_samples.png` và `fig_gt_vs_pred_comparison.png`).

In [ ]:
import os
import glob
import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import model_tf

print(">> Đang nạp mô hình RGB-Only để kiểm tra trực quan...")
try:
    model = tf.keras.models.load_model(DEST_CKPT, custom_objects={"CoordinateAttention": model_tf.CoordinateAttention, "AnchorFreeCenterHead": model_tf.AnchorFreeCenterHead}, compile=False)
except Exception:
    model = model_tf.build_detection_model(input_shape=(640, 640, 9), fpn_dim=128)
    model.load_weights(DEST_CKPT)

df = pd.read_csv(EVAL_CSV)
pos_candidates = df[df["exist"] == 1]
pos_samples = pos_candidates.sample(min(4, len(pos_candidates)), random_state=42).reset_index(drop=True)

fig, axes = plt.subplots(2, 2, figsize=(16, 11), dpi=300)
axes = axes.flatten()

def load_norm_frame(path):
    raw = cv2.imread(path)
    if raw is None: return np.zeros((640, 640, 3), dtype=np.float32)
    raw = cv2.cvtColor(raw, cv2.COLOR_BGR2RGB)
    im = cv2.resize(raw, (640, 640))
    return im.astype(np.float32) / 255.0

for i, row in pos_samples.iterrows():
    p_vis = os.path.join(VOC_ROOT, row["vis_t"])
    img_v = cv2.cvtColor(cv2.imread(p_vis), cv2.COLOR_BGR2RGB)
    h, w = img_v.shape[:2]

    t_in = np.concatenate([
        load_norm_frame(os.path.join(VOC_ROOT, row["vis_tm1"])),
        load_norm_frame(p_vis),
        load_norm_frame(os.path.join(VOC_ROOT, row["vis_tp1"]))
    ], axis=-1)[np.newaxis, ...]

    preds = model(t_in, training=False)
    scores, bboxes = model_tf.decode_detections(preds["heatmap"], preds["offset"], preds["size"])
    conf = float(scores[0].numpy())
    pred_box = bboxes[0].numpy() * [w, h, w, h]
    gt_box   = [row["xmin"], row["ymin"], row["xmax"], row["ymax"]]

    ax = axes[i]
    ax.imshow(img_v)
    ax.set_title(f"ANTI-UAV RGB - Mẫu {i+1} [{row['sequence_id']} | Frame {row['frame_idx']}]: GT (Xanh Lá) vs Pred (Đỏ, Conf={conf*100:.1f}%)",
                 fontsize=10, fontweight="bold")
    ax.axis("off")

    bw_gt, bh_gt = gt_box[2] - gt_box[0], gt_box[3] - gt_box[1]
    ax.add_patch(patches.Rectangle((gt_box[0], gt_box[1]), bw_gt, bh_gt,
                                   linewidth=2, edgecolor="#00ff00", facecolor="none", linestyle="--"))
    ax.text(gt_box[0], max(0, gt_box[1] - 8), f"GT ({bw_gt:.0f}x{bh_gt:.0f})", color="white", fontsize=8, fontweight="bold",
            bbox=dict(facecolor="#00AA00", edgecolor="none", pad=1.5))

    if conf >= 0.40:
        bw_pr, bh_pr = pred_box[2] - pred_box[0], pred_box[3] - pred_box[1]
        ax.add_patch(patches.Rectangle((pred_box[0], pred_box[1]), bw_pr, bh_pr,
                                       linewidth=2.5, edgecolor="red", facecolor="none"))
        ax.text(pred_box[0], max(0, pred_box[1] - 8), f"Drone {conf*100:.1f}%",
                color="white", fontsize=8, fontweight="bold",
                bbox=dict(facecolor="red", edgecolor="none", pad=1.5))

plt.tight_layout()
out_v_path = "/kaggle/working/report_figures/fig_visual_test_samples.png"
plt.savefig(out_v_path, bbox_inches="tight")
plt.savefig("/kaggle/working/checkpoints/fig_visual_predictions.png", bbox_inches="tight")
plt.show()
print(f">> [x] ĐÃ XUẤT ẢNH DỰ ĐOÁN TRỰC QUAN TẠI: {out_v_path}")

In [ ]:
# So sánh định lượng chi tiết kèm chỉ số IoU và NWD từng mẫu (Toàn cảnh & Cận cảnh Drone)
def calc_iou(b1, b2):
    x1, y1 = max(b1[0], b2[0]), max(b1[1], b2[1])
    x2, y2 = min(b1[2], b2[2]), min(b1[3], b2[3])
    inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    a1 = max(0.0, b1[2] - b1[0]) * max(0.0, b1[3] - b1[1])
    a2 = max(0.0, b2[2] - b2[0]) * max(0.0, b2[3] - b2[1])
    union = a1 + a2 - inter
    return inter / union if union > 1e-7 else 0.0

def calc_nwd(b1, b2, c_norm=0.02):
    cx1, cy1 = (b1[0] + b1[2]) / 2.0, (b1[1] + b1[3]) / 2.0
    w1, h1   = max(b1[2] - b1[0], 1e-4), max(b1[3] - b1[1], 1e-4)
    cx2, cy2 = (b2[0] + b2[2]) / 2.0, (b2[1] + b2[3]) / 2.0
    w2, h2   = max(b2[2] - b2[0], 1e-4), max(b2[3] - b2[1], 1e-4)
    d2 = (cx1 - cx2)**2 + (cy1 - cy2)**2 + ((w1 - w2)**2 + (h1 - h2)**2) / 4.0
    return float(np.exp(-np.sqrt(max(d2, 1e-7)) / c_norm))

fig, axes = plt.subplots(len(pos_samples), 2, figsize=(16, 4.5 * len(pos_samples)), dpi=300)

for i, row in pos_samples.iterrows():
    p_vis = os.path.join(VOC_ROOT, row["vis_t"])
    raw_vis = cv2.imread(p_vis)
    if raw_vis is None: continue
    img_vis_rgb = cv2.cvtColor(raw_vis, cv2.COLOR_BGR2RGB)
    h_orig, w_orig = img_vis_rgb.shape[:2]

    tensor_9ch = np.concatenate([
        load_norm_frame(os.path.join(VOC_ROOT, row["vis_tm1"])),
        load_norm_frame(p_vis),
        load_norm_frame(os.path.join(VOC_ROOT, row["vis_tp1"]))
    ], axis=-1)[np.newaxis, ...]

    preds = model(tensor_9ch, training=False)
    scores, bboxes = model_tf.decode_detections(preds["heatmap"], preds["offset"], preds["size"])
    conf = float(scores[0].numpy())
    norm_pred_box = bboxes[0].numpy()

    pred_box = norm_pred_box * [w_orig, h_orig, w_orig, h_orig]
    gt_box   = [row["xmin"], row["ymin"], row["xmax"], row["ymax"]]
    norm_gt_box = [gt_box[0] / w_orig, gt_box[1] / h_orig, gt_box[2] / w_orig, gt_box[3] / h_orig]

    iou_val = calc_iou(gt_box, pred_box)
    nwd_val = calc_nwd(norm_gt_box, norm_pred_box)

    # Cột 1: Toàn cảnh Full HD
    ax_v = axes[i, 0]
    ax_v.imshow(img_vis_rgb)
    title_text = f"ANTI-UAV RGB - Mẫu {i+1}: GT (Xanh Lá) vs Dự Đoán (Đỏ)\nConf: {conf*100:.1f}% | IoU: {iou_val:.2f} | NWD: {nwd_val:.2f}"
    ax_v.set_title(title_text, fontsize=11, fontweight="bold", pad=8)
    ax_v.axis("off")

    bw_gt = gt_box[2] - gt_box[0]
    bh_gt = gt_box[3] - gt_box[1]
    ax_v.add_patch(patches.Rectangle((gt_box[0], gt_box[1]), bw_gt, bh_gt,
                                     linewidth=2, edgecolor="#00FF00", linestyle="--", facecolor="none"))
    ax_v.text(gt_box[0], max(0, gt_box[1] - 8), f"GT ({bw_gt:.0f}x{bh_gt:.0f})", color="white", fontsize=9, fontweight="bold",
              bbox=dict(facecolor="#00AA00", edgecolor="none", pad=1.5))

    if conf >= 0.40:
        bw_pr = pred_box[2] - pred_box[0]
        bh_pr = pred_box[3] - pred_box[1]
        ax_v.add_patch(patches.Rectangle((pred_box[0], pred_box[1]), bw_pr, bh_pr,
                                         linewidth=2.5, edgecolor="red", facecolor="none"))
        ax_v.text(pred_box[0], max(0, pred_box[1] - 8), f"Drone {conf*100:.0f}%",
                  color="white", fontsize=9, fontweight="bold",
                  bbox=dict(facecolor="red", edgecolor="none", pad=1.5))

    # Cột 2: Zoom cận cảnh mục tiêu (Zoom Inset)
    ax_z = axes[i, 1]
    pad = max(bw_gt, bh_gt) * 2.5
    cx, cy = (gt_box[0] + gt_box[2]) / 2.0, (gt_box[1] + gt_box[3]) / 2.0
    z_x1, z_x2 = max(0, int(cx - pad)), min(w_orig, int(cx + pad))
    z_y1, z_y2 = max(0, int(cy - pad)), min(h_orig, int(cy + pad))

    crop = img_vis_rgb[z_y1:z_y2, z_x1:z_x2]
    ax_z.imshow(crop)
    ax_z.set_title(f"Mẫu {i+1} [Cận Cảnh Drone Phóng Đại]: So Sánh GT vs Pred", fontsize=11, fontweight="bold", pad=8)
    ax_z.axis("off")

    ax_z.add_patch(patches.Rectangle((gt_box[0] - z_x1, gt_box[1] - z_y1), bw_gt, bh_gt,
                                     linewidth=2.5, edgecolor="#00FF00", linestyle="--", facecolor="none"))
    if conf >= 0.40:
        bw_pr = pred_box[2] - pred_box[0]
        bh_pr = pred_box[3] - pred_box[1]
        ax_z.add_patch(patches.Rectangle((pred_box[0] - z_x1, pred_box[1] - z_y1), bw_pr, bh_pr,
                                         linewidth=2.5, edgecolor="red", facecolor="none"))

plt.tight_layout()
out_cmp_path = "/kaggle/working/report_figures/fig_gt_vs_pred_comparison.png"
plt.savefig(out_cmp_path, bbox_inches="tight", dpi=300)
plt.savefig("/kaggle/working/fig_gt_vs_pred_comparison.png", bbox_inches="tight", dpi=300)
plt.show()
print(f">> [x] ĐÃ XUẤT ẢNH SO SÁNH CHẤT LƯỢNG CAO TẠI: {out_cmp_path}")

---
## PHẦN 4: RENDER BỘ VIDEO DEMO TÁC CHIẾN THỜI GIAN THỰC (3 HẠNG MỤC x 3 VIDEO = 9 VIDEO)
Hệ thống tự động phân tích và chọn **9 chuỗi video đại diện từ tập kiểm thử (Test Set)**, chia đều vào **3 phân khúc kích thước Drone** (mỗi phân khúc gồm 3 video độc lập) nhằm cung cấp các góc nhìn đa dạng và giúp người dùng dễ dàng so sánh, lựa chọn video tối ưu nhất:

1. **Phân khúc 1: Drone To (Large Scale - 3 Video)**
   * Bounding box có diện tích $\ge 96^2\text{ px}^2$ (mục tiêu kích thước lớn, nhìn rõ cánh quạt và kết cấu khung thân).
   * Đánh giá độ chính xác bao phủ Bounding Box (IoU) và khả năng bám bắt chi tiết khi mục tiêu ở cự ly gần.
2. **Phân khúc 2: Drone Trung Bình (Medium Scale - 3 Video)**
   * Diện tích nằm trong khoảng $32^2 \le \text{Area} < 96^2\text{ px}^2$ (kích thước tầm trung).
   * Đại diện cho các tình huống bay cơ động điển hình ở cự ly trung bình với vận tốc góc biến thiên liên tục.
3. **Phân khúc 3: Drone Nhỏ (Small / Tiny Scale - 3 Video)**
   * Diện tích $< 32^2\text{ px}^2$ (mục tiêu siêu nhỏ, kích thước cực kỳ khó phân biệt trên nền trời hoặc vật cản).
   * Chứng minh hiệu năng vượt trội của hàm mất mát Normalized Wasserstein Distance (NWD) trong việc định vị micro-drone.

**Quy trình xử lý tác chiến:**
* Tích hợp **Bộ lọc quán tính quỹ đạo (`TrajectoryEMAFilter`)** làm mượt tọa độ tâm, chống giật rung khung hình và duy trì khóa mục tiêu khi Drone cơ động đổi hướng hoặc bị che khuất ngắn hạn.
* Hiển thị giao diện **HUD chiến thuật** trên **1 Camera Visible RGB duy nhất** với tiêu đề chuẩn **"ANTI-UAV RGB"** (đã lược bỏ Cam 2 Heatmap Motion).
* Toàn bộ 9 video được mã hóa chuẩn **H.264 (libx264, yuv420p)** để phát trực tiếp mượt mà trên trình duyệt web.

In [ ]:
import glob
import os
import numpy as np
import pandas as pd

# 1. Đọc dữ liệu tập kiểm định / kiểm thử và lọc các mẫu có tồn tại Drone
df_test = pd.read_csv(EVAL_CSV)
pos_test = df_test[df_test["exist"] == 1].copy()

# 2. Thống kê kích thước (Bounding Box Area) trung bình theo từng chuỗi video
seq_stats = []
for seq_id, group in pos_test.groupby("sequence_id"):
    w = group["xmax"] - group["xmin"]
    h = group["ymax"] - group["ymin"]
    areas = w * h
    seq_stats.append({
        "sequence_id": seq_id,
        "frame_count": len(group),
        "mean_area": float(areas.mean()),
        "mean_w": float(w.mean()),
        "mean_h": float(h.mean()),
        "max_area": float(areas.max()),
        "min_area": float(areas.min())
    })

df_seq = pd.DataFrame(seq_stats)

# Ưu tiên các chuỗi có số khung hình quan sát đủ dài (tối thiểu 25 frames)
valid_seqs = df_seq[df_seq["frame_count"] >= 25]
if len(valid_seqs) < 9:
    valid_seqs = df_seq[df_seq["frame_count"] >= 10]
if len(valid_seqs) < 9:
    valid_seqs = df_seq

sorted_seqs = valid_seqs.sort_values(by="mean_area").reset_index(drop=True)
n_total = len(sorted_seqs)

# 3. Phân bổ mỗi phân khúc 3 video độc lập để người dùng dễ dàng đối sánh và lựa chọn
chosen_seqs = {"large": [], "medium": [], "small": []}
used_ids = set()

# A. Phân khúc Drone To (Large): 3 chuỗi có diện tích lớn nhất (ưu tiên Area >= 96^2)
large_cands = sorted_seqs[sorted_seqs["mean_area"] >= (96 * 96)]
if len(large_cands) >= 3:
    top_large = large_cands.sort_values(by="mean_area", ascending=False).head(3)
else:
    top_large = sorted_seqs.sort_values(by="mean_area", ascending=False).head(min(3, n_total))

for _, r in top_large.iterrows():
    chosen_seqs["large"].append(r.to_dict())
    used_ids.add(r["sequence_id"])

# B. Phân khúc Drone Nhỏ (Small): 3 chuỗi có diện tích nhỏ nhất (ưu tiên Area < 32^2)
rem_for_small = sorted_seqs[~sorted_seqs["sequence_id"].isin(used_ids)]
small_cands = rem_for_small[rem_for_small["mean_area"] < (32 * 32)]
if len(small_cands) >= 3:
    top_small = small_cands.sort_values(by="mean_area", ascending=True).head(3)
else:
    top_small = rem_for_small.sort_values(by="mean_area", ascending=True).head(min(3, len(rem_for_small)))

for _, r in top_small.iterrows():
    chosen_seqs["small"].append(r.to_dict())
    used_ids.add(r["sequence_id"])

# C. Phân khúc Drone Trung bình (Medium): 3 chuỗi ở khoảng giữa (32^2 <= Area < 96^2)
rem_for_med = sorted_seqs[~sorted_seqs["sequence_id"].isin(used_ids)]
med_cands = rem_for_med[(rem_for_med["mean_area"] >= (32 * 32)) & (rem_for_med["mean_area"] < (96 * 96))]
if len(med_cands) >= 3:
    top_med = med_cands.sort_values(by="frame_count", ascending=False).head(3)
else:
    if len(rem_for_med) >= 3:
        mid_idx = len(rem_for_med) // 2
        top_med = rem_for_med.iloc[max(0, mid_idx - 1): max(0, mid_idx - 1) + 3]
    else:
        top_med = rem_for_med

for _, r in top_med.iterrows():
    chosen_seqs["medium"].append(r.to_dict())
    used_ids.add(r["sequence_id"])

category_configs = [
    ("PHÂN KHÚC 1: DRONE TO (LARGE SCALE)", "large", "TO (LARGE)"),
    ("PHÂN KHÚC 2: DRONE TRUNG BÌNH (MEDIUM SCALE)", "medium", "TRUNG BINH (MEDIUM)"),
    ("PHÂN KHÚC 3: DRONE NHỎ (SMALL / TINY SCALE)", "small", "NHO (SMALL)"),
]

print("=" * 85)
print("DANH SÁCH 9 CHUỖI VIDEO ĐƯỢC CHỌN (MỖI HẠNG MỤC 3 VIDEO ĐỂ LỰA CHỌN):")
print("=" * 85)

render_tasks = []
for cat_title, cat_key, hud_name in category_configs:
    items = chosen_seqs[cat_key]
    print(f"\n>> {cat_title} (Số lượng: {len(items)} video):")
    for idx, info in enumerate(items, 1):
        seq_name = info["sequence_id"]
        n_f = int(info["frame_count"])
        w_avg = info["mean_w"]
        h_avg = info["mean_h"]
        area_avg = info["mean_area"]
        out_file = f"/kaggle/working/rendered_videos/test_demo_{cat_key}_{idx}_{seq_name}.mp4"
        render_tasks.append((cat_title, cat_key, idx, seq_name, out_file, info))
        print(f"   [{idx}/3] Sequence: {seq_name:<20} | Frames: {n_f:3d} | Kích thước TB: {w_avg:5.1f}x{h_avg:5.1f} px | Diện tích: {area_avg:7.1f} px^2")

# 4. Tiến hành Render lần lượt toàn bộ 9 video tác chiến
os.makedirs("/kaggle/working/rendered_videos", exist_ok=True)
total_tasks = len(render_tasks)
print("\n" + "=" * 85)
print(f"BẮT ĐẦU RENDER {total_tasks} VIDEO TÁC CHIẾN CHO CẢ 3 HẠNG MỤC DRONE...")
print("=" * 85)

for i, (cat_title, cat_key, idx, seq_name, out_file, info) in enumerate(render_tasks, 1):
    print(f"\n>> [{i}/{total_tasks}] ĐANG RENDER [{cat_key.upper()} - VIDEO {idx}/3]: {seq_name}...")
    cmd = (
        f'python infer_video.py '
        f'--voc_root "{VOC_ROOT}" '
        f'--checkpoint "{DEST_CKPT}" '
        f'--seq_id {seq_name} '
        f'--output "{out_file}" '
        f'--modality rgb '
        f'--max_frames 200 '
        f'--conf_thresh 0.40 '
    )
    os.system(cmd)

print("\n" + "=" * 85)
print(f"HOÀN THÀNH RENDER XUẤT SẮC TOÀN BỘ {total_tasks} VIDEO Ở CẢ 3 HẠNG MỤC!")
print("=" * 85)

In [ ]:
from IPython.display import HTML, display
from base64 import b64encode
import glob
import os

category_meta = [
    ("large", "HẠNG MỤC 1: PHÂN KHÚC DRONE TO (LARGE SCALE)", "#d93025"),
    ("medium", "HẠNG MỤC 2: PHÂN KHÚC DRONE TRUNG BÌNH (MEDIUM SCALE)", "#f29900"),
    ("small", "HẠNG MỤC 3: PHÂN KHÚC DRONE NHỎ (SMALL / TINY SCALE)", "#1a73e8"),
]

all_rendered = glob.glob("/kaggle/working/rendered_videos/test_demo_*.mp4")
print(f"Tổng số video đã render tìm thấy trong thư mục: {len(all_rendered)} files\n")

for cat_key, cat_title, border_color in category_meta:
    cat_videos = sorted([v for v in all_rendered if f"_{cat_key}_" in os.path.basename(v)])
    
    display(HTML(f"""
        <div style="background: #f8f9fa; border-left: 5px solid {border_color}; padding: 12px 18px; margin-top: 30px; margin-bottom: 15px; border-radius: 4px;">
            <h3 style="color: {border_color}; margin: 0 0 5px 0; font-size: 18px;">{cat_title}</h3>
            <p style="margin: 0; color: #5f6368; font-size: 13px;">Gồm {len(cat_videos)} video ứng viên khác nhau để bạn xem và lựa chọn video ưng ý nhất.</p>
        </div>
    """))
    
    if not cat_videos:
        print(f"   [!] Chưa có video nào được render cho hạng mục {cat_key}!")
        continue
        
    for v_idx, v_path in enumerate(cat_videos, 1):
        v_name = os.path.basename(v_path)
        v_size_mb = os.path.getsize(v_path) / (1024 * 1024)
        print(f"   >> [{cat_key.upper()} - LỰA CHỌN {v_idx}/3]: {v_name} ({v_size_mb:.2f} MB)")
        
        mp4_bytes = open(v_path, 'rb').read()
        data_url = "data:video/mp4;base64," + b64encode(mp4_bytes).decode()
        display(HTML(f"""
            <div style="margin-bottom: 25px; padding-left: 10px;">
                <div style="font-weight: bold; color: #202124; margin-bottom: 6px; font-size: 14px;">
                    Lựa chọn {v_idx}/3 ({cat_key.upper()}): <span style="font-family: monospace; color: #1a73e8;">{v_name}</span> ({v_size_mb:.2f} MB)
                </div>
                <video width="850" controls style="border-radius: 8px; box-shadow: 0 4px 12px rgba(0,0,0,0.25); border: 1px solid #dadce0;">
                    <source src="{data_url}" type="video/mp4">
                </video>
            </div>
        """))

---
## PHẦN 5: TỔNG KẾT DANH MỤC TÀI LIỆU BÁO CÁO (OUTPUT ARTIFACTS)
Kiểm tra tính toàn vẹn của tất cả tài liệu, hình ảnh báo cáo và video để sẵn sàng tải về máy.

In [ ]:
import os
import glob

print("=" * 80)
print(" KIỂM TRA ĐẦY ĐỦ CÁC TÀI LIỆU BÁO CÁO (OUTPUT ARTIFACTS) TẠI /kaggle/working:")
print("=" * 80)

def format_size(bytes_val):
    for unit in ['B', 'KB', 'MB', 'GB']:
        if bytes_val < 1024:
            return f"{bytes_val:.2f} {unit}"
        bytes_val /= 1024
    return f"{bytes_val:.2f} TB"

expected_items = {
    "[1] CHECKPOINTS & TRỌNG SỐ MÔ HÌNH": [
        "/kaggle/working/checkpoints/best_uav_model.keras",
        "/kaggle/working/checkpoints/training_history.csv",
    ],
    "[2] HÌNH ẢNH KHOA HỌC BÁO CÁO (300 DPI)": [
        "/kaggle/working/report_figures/fig1_rgbt_temporal_triplet.png",
        "/kaggle/working/report_figures/fig2_drone_scale_distribution.png",
        "/kaggle/working/report_figures/loss_curve.png",
        "/kaggle/working/report_figures/pr_curve.png",
        "/kaggle/working/report_figures/fig_visual_test_samples.png",
        "/kaggle/working/report_figures/fig_gt_vs_pred_comparison.png",
    ],
    "[3] VIDEO DEMO TÁC CHIẾN (H.264 + HUD TACTICAL)": glob.glob("/kaggle/working/rendered_videos/*.mp4")
}

all_found = True
for section, file_list in expected_items.items():
    print(f"\n{section}:")
    if not file_list:
        print("   [!] Chưa có file nào trong mục này!")
        all_found = False
    for f in file_list:
        if os.path.exists(f):
            sz = os.path.getsize(f)
            print(f"   [x] {os.path.basename(f):<35} ({format_size(sz)}) -> {f}")
        else:
            print(f"   [ ] THIẾU: {f}")
            all_found = False

print("\n" + "=" * 80)
if all_found:
    print("  HOÀN TOÀN ĐẦY ĐỦ TẤT CẢ FILE BÁO CÁO! SẴN SÀNG TẢI VỀ HOẶC NỘP BÀI.")
else:
    print(" CẢNH BÁO: MỘT SỐ FILE CHƯA ĐƯỢC TẠO. HÃY KIỂM TRA LẠI CÁC BƯỚC TRÊN.")
print("=" * 80)